# 1. Data Download & EDA

This section covers downloading Pune OpenAQ AQI data, cleaning it, exploring it, and engineering features for our LSTM model.

## 1.1 Download Pune OpenAQ AQI Data
We can download data from `api.data.gov.in` using an API key or manually upload CSVs from the OpenAQ portal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import holidays
from io import StringIO

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ---------------------------------------------------------
# Option A: API Fetch (api.data.gov.in)
# ---------------------------------------------------------
# Example code for using the API (Requires YOUR_API_KEY)
'''
api_key = "YOUR_API_KEY_HERE"
url = f"https://api.data.gov.in/resource/3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69?api-key={api_key}&format=csv&limit=1000"
response = requests.get(url)
if response.status_code == 200:
    df_api = pd.read_csv(StringIO(response.text))
    display(df_api.head())
'''

# ---------------------------------------------------------
# Option B: Manual Upload Instructions
# ---------------------------------------------------------
# 1. Visit: https://app.openaqccr.com/ccr/#/caaqm-dashboard-all/caaqm-landing
# 2. Select State: Maharashtra, City: Pune, Station: (e.g., Shivajinagar)
# 3. Select Parameters: PM2.5, PM10, NO2, SO2, CO, AQI
# 4. Select Date Range and Export as CSV.
# 5. Upload the CSV to Colab and update the path below:
# df = pd.read_csv('pune_openaq_data.csv')

# ---------------------------------------------------------
# For notebook demonstration, we generate a dataset
# mimicking hourly data for Pune from 2022 to 2024.
# ---------------------------------------------------------
dates = pd.date_range(start='2022-01-01', end='2024-03-01', freq='h')
np.random.seed(42)

df = pd.DataFrame({
    'timestamp': dates,
    'station_id': 'PUNE_SHIVAJINAGAR',
    'pm25': np.random.normal(50, 25, len(dates)) + np.sin(np.arange(len(dates)) * (2 * np.pi / 24)) * 10 + np.sin(np.arange(len(dates)) * (2 * np.pi / (24 * 365))) * 30,
    'pm10': np.random.normal(100, 40, len(dates)),
    'no2': np.random.normal(30, 15, len(dates)) + np.cos(np.arange(len(dates)) * (2 * np.pi / 24)) * 5,
    'so2': np.random.normal(10, 5, len(dates)),
    'co': np.random.normal(1.0, 0.4, len(dates)),
    'aqi': np.random.normal(120, 50, len(dates))
})

# Add some random gaps and errors to require cleaning
df.loc[np.random.choice(df.index, 500, replace=False), 'pm25'] = np.nan
df.loc[np.random.choice(df.index, 50, replace=False), 'pm25'] = 800  # Error: too high
df.loc[np.random.choice(df.index, 50, replace=False), 'pm25'] = -20  # Error: negative

print(f"Raw Data Shape: {df.shape}")
display(df.head())

## 1.2 Data Cleaning
- Parse timestamps to datetime, set as index
- Show missing data heatmap using seaborn
- Flag and remove readings where pm25 > 500 or pm25 < 0 (sensor errors)
- Resample to hourly (mean), fill gaps <= 3 hours with linear interpolation

In [ ]:
# Parse timestamps to datetime, set as index
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

# Show missing data heatmap using seaborn
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Data Heatmap Before Cleaning')
plt.tight_layout()
plt.show()

# Flag and remove readings where pm25 > 500 or pm25 < 0 (sensor errors)
pm25_errors = (df['pm25'] > 500) | (df['pm25'] < 0)
print(f"Found {pm25_errors.sum()} PM2.5 sensor errors (<0 or >500). Setting them to NaN.")
df.loc[pm25_errors, 'pm25'] = np.nan

# Resample to hourly (mean), fill gaps <= 3 hours with linear interpolation
numeric_cols = ['pm25', 'pm10', 'no2', 'so2', 'co', 'aqi']
df_resampled = df[numeric_cols].resample('h').mean()
df_resampled = df_resampled.interpolate(method='linear', limit=3)

print("\nMissing values after cleaning and interpolation:")
print(df_resampled.isnull().sum())

## 1.3 EDA Plots
Visualizing hourly, weekly, and monthly patterns, along with feature correlations.

In [ ]:
# Prepare features for plotting
eda_df = df_resampled.copy()
eda_df['hour'] = eda_df.index.hour
eda_df['day_of_week'] = eda_df.index.dayofweek
eda_df['month'] = eda_df.index.month
eda_df['year'] = eda_df.index.year

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
plt.subplots_adjust(hspace=0.4, wspace=0.2)

# 1. Hourly AQI pattern (average by hour of day)
sns.lineplot(data=eda_df, x='hour', y='aqi', ax=axes[0, 0], marker='o', errorbar=None)
axes[0, 0].set_title('Hourly AQI Pattern - Rush Hour Peaks')
axes[0, 0].set_xticks(range(0, 24))
axes[0, 0].set_xlabel('Hour of Day')

# 2. Weekly pattern (average by day of week)
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
sns.lineplot(data=eda_df, x='day_of_week', y='aqi', ax=axes[0, 1], marker='o', errorbar=None)
axes[0, 1].set_title('Weekly AQI Pattern')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(days)
axes[0, 1].set_xlabel('Day of Week')

# 3. Monthly trend for 2022–2024
monthly_trend = eda_df.groupby(['year', 'month'])['aqi'].mean().reset_index()
monthly_trend['date'] = pd.to_datetime(monthly_trend[['year', 'month']].assign(day=1))
sns.lineplot(data=monthly_trend, x='date', y='aqi', ax=axes[1, 0], marker='o')
axes[1, 0].set_title('Monthly AQI Trend (2022 - 2024)')
axes[1, 0].set_xlabel('Date')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. PM2.5 vs NO2 scatter plot (vehicular correlation)
sns.scatterplot(data=eda_df, x='no2', y='pm25', alpha=0.3, ax=axes[1, 1], color='coral')
axes[1, 1].set_title('PM2.5 vs NO2 Scatter Plot (Vehicular Correlation)')

# 5. Correlation heatmap of all pollutants
corr = df_resampled[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=axes[2, 0], square=True)
axes[2, 0].set_title('Correlation Heatmap')

# Hide the empty subplot
axes[2, 1].axis('off')

plt.show()

## 1.4 Feature Engineering
Creating cyclical time features, weekday/weekend flags, lag features, and rolling statistics.

In [ ]:
feat_df = df_resampled.copy()

# 1. Cyclical hour features
hour = feat_df.index.hour
feat_df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
feat_df['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# 2. Day of week, weekend, and holiday features
feat_df['day_of_week'] = feat_df.index.dayofweek
feat_df['is_weekend'] = (feat_df['day_of_week'] >= 5).astype(int)

# Use Indian holidays list
in_holidays = holidays.India(years=feat_df.index.year.unique().tolist())
feat_df['is_holiday'] = feat_df.index.normalize().map(lambda x: x.date() in in_holidays).astype(int)

# 3. Lagged features (1h, 3h, 6h, 24h) for PM2.5
feat_df['pm25_lag_1h'] = feat_df['pm25'].shift(1)
feat_df['pm25_lag_3h'] = feat_df['pm25'].shift(3)
feat_df['pm25_lag_6h'] = feat_df['pm25'].shift(6)
feat_df['pm25_lag_24h'] = feat_df['pm25'].shift(24)

# 4. Rolling means (3h, 6h) for PM2.5
feat_df['rolling_mean_3h'] = feat_df['pm25'].rolling(window=3).mean()
feat_df['rolling_mean_6h'] = feat_df['pm25'].rolling(window=6).mean()

# Drop rows with NaNs caused by lagging/rolling
feat_df.dropna(inplace=True)

print(f"Final Feature DataFrame Shape: {feat_df.shape}")
print("\n--- Data Types ---")
print(feat_df.dtypes)
print("\n--- First 5 Rows ---")
display(feat_df.head())

# 2. LSTM Model Training
This section covers sequence preparation, model definition, training with WandB, and evaluation.

## 2.1 Feature Integration & Sequence Preparation
We integrate meteorological and traffic features, split the data chronologically (70/15/15), and scale the inputs/outputs.

In [ ]:
import os
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Setup Inputs and Targets
# Since the first section didn't include some external features requested, we mock them here
# to represent integrating with a Weather API (Open-Meteo) and Traffic API.
np.random.seed(42)
n_samples = len(feat_df)
feat_df['wind_speed'] = np.random.uniform(0, 15, n_samples)
feat_df['wind_dir_sin'] = np.sin(np.random.uniform(0, 2*np.pi, n_samples))
feat_df['wind_dir_cos'] = np.cos(np.random.uniform(0, 2*np.pi, n_samples))
feat_df['humidity'] = np.random.uniform(30, 90, n_samples)
feat_df['temp'] = np.random.uniform(15, 40, n_samples)
feat_df['traffic_index'] = np.random.uniform(0, 10, n_samples) + feat_df['hour_sin'] * 2

feature_cols = [
    'pm25', 'no2', 'wind_speed', 'wind_dir_sin', 'wind_dir_cos',
    'humidity', 'temp', 'traffic_index', 'hour_sin', 'hour_cos', 'day_of_week'
]

# Create targets for multiple horizons: +1h, +3h, +6h, +12h, +24h
target_horizons = [1, 3, 6, 12, 24]
for h in target_horizons:
    feat_df[f'aqi_target_{h}h'] = feat_df['aqi'].shift(-h)

# Drop rows where targets are NaN due to shifting
feat_df = feat_df.dropna()
target_cols = [f'aqi_target_{h}h' for h in target_horizons]

print("Feature Dataframe Shape (After Shifting Targets):", feat_df.shape)

# 2. Chronological Train/Val/Test Split (70 / 15 / 15)
n = len(feat_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = feat_df.iloc[:train_end]
val_df = feat_df.iloc[train_end:val_end]
test_df = feat_df.iloc[val_end:]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# 3. Scaling
scaler_X = StandardScaler()
scaler_y = MinMaxScaler()

train_X_scaled = scaler_X.fit_transform(train_df[feature_cols])
val_X_scaled = scaler_X.transform(val_df[feature_cols])
test_X_scaled = scaler_X.transform(test_df[feature_cols])

train_y_scaled = scaler_y.fit_transform(train_df[target_cols])
val_y_scaled = scaler_y.transform(val_df[target_cols])
test_y_scaled = scaler_y.transform(test_df[target_cols])

# 4. Create sequences of length 24 (past 24 hours)
def create_sequences(X, y, seq_length=24):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i:(i + seq_length)])
        ys.append(y[i + seq_length])
    return np.array(Xs), np.array(ys)

SEQ_LEN = 24
X_train, y_train = create_sequences(train_X_scaled, train_y_scaled, SEQ_LEN)
X_val, y_val = create_sequences(val_X_scaled, val_y_scaled, SEQ_LEN)
X_test, y_test = create_sequences(test_X_scaled, test_y_scaled, SEQ_LEN)

print(f"X_train sequence shape: {X_train.shape}, y_train shape: {y_train.shape}")

## 2.2 Model Architecture & Training
Using TensorFlow/Keras to build a Bidirectional LSTM.

In [ ]:
!pip install -q wandb
import wandb
from wandb.integration.keras import WandbMetricsLogger
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Init Wandb
# wandb.login() # Manual login if running the first time
wandb.init(project="expoair-lstm", name="bi-lstm-aqi-multi-horizon")

# Architecture: Input(24, 11) -> BiLSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16, relu) -> Dense(5)
model = Sequential([
    Input(shape=(24, 11)),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(5) # Predicts +1h, +3h, +6h, +12h, +24h AQI simultaneously
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
wandb_logger = WandbMetricsLogger()

# Train Model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop, reduce_lr, wandb_logger],
    verbose=1
)

wandb.finish()

## 2.3 Evaluation
Testing the forecasting models with visualizations of Real vs Pred and error histograms.

In [ ]:
# Predictions on Test set
y_pred_scaled = model.predict(X_test)

# Inverse Scale outputs
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_test_inv = scaler_y.inverse_transform(y_test)

# 1. Compute RMSE, MAE, R^2 per horizon
horizons_labels = ['+1h', '+3h', '+6h', '+12h', '+24h']
metrics = []

for i, h in enumerate(horizons_labels):
    true_vals = y_test_inv[:, i]
    pred_vals = y_pred[:, i]
    
    rmse = np.sqrt(mean_squared_error(true_vals, pred_vals))
    mae = mean_absolute_error(true_vals, pred_vals)
    r2 = r2_score(true_vals, pred_vals)
    metrics.append({'Horizon': h, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

display(pd.DataFrame(metrics))

# 2. Plot Predicted vs Actual for a slice of the test set
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_length = 200 # First 200 hours

axes[0].plot(y_test_inv[:plot_length, 0], label='Actual (+1h)', marker='.', linestyle='dashed')
axes[0].plot(y_pred[:plot_length, 0], label='Predicted (+1h)', marker='.', alpha=0.8)
axes[0].set_title('Test Set: Demonstrating +1h AQI Forecast')
axes[0].legend()

axes[1].plot(y_test_inv[:plot_length, 4], label='Actual (+24h)', marker='.', linestyle='dashed')
axes[1].plot(y_pred[:plot_length, 4], label='Predicted (+24h)', marker='.', alpha=0.8)
axes[1].set_title('Test Set: Demonstrating +24h AQI Forecast')
axes[1].legend()
plt.show()

# 3. Error Distribution Histogram
errors = y_pred - y_test_inv
plt.figure(figsize=(10, 5))
for i, h in enumerate(horizons_labels):
    sns.kdeplot(errors[:, i], label=f'Error {h}', fill=True, alpha=0.3)
plt.title('Error Distribution for All Forecasting Horizons')
plt.xlabel('Prediction Error (AQI Off By)')
plt.legend()
plt.xlim(-100, 100)
plt.show()

## 2.4 Save Models & Scalers
Save the generated models and scaling objects to be used directly by the FastAPI backend.

In [ ]:
os.makedirs("models_saved", exist_ok=True)

# Save the Keras architecture and best weights
model.save("models_saved/lstm_aqi.keras")

# We save a dictionary containing both scalers to the requested scaler.pkl location
scalers_dict = {
    'X': scaler_X, 
    'y': scaler_y
}
joblib.dump(scalers_dict, "models_saved/scaler.pkl")

print("Exported `models_saved/lstm_aqi.h5` and `models_saved/scaler.pkl` successfully.")